# MirrorTopology Step 1 — Phase D4C-0a（D-2W）：登録済み D-2 W₂ 位置 bank による 9 formal W₂ case の評価と typed record の publication（notebook v0.2）
受入れ済みの D-2 run（E2／E7／E8 の `out/d2`：`cfg<id>_w2` 27 unit＝第 1 波の 3 position × 3 size × 3 family，matched primary の raw T，f64＋f32）を **formal intake**（bytes／配列 SHA の再検証・spec 同一性・root SHA＝登録 D-1 共分散の matched root・登録 shared-null の whitening 値・manifest SHA＝登録 D-2 ledger の unit）で読み，rules §10 の W₂ 比較（n_sub 2000／5000・3 seed・B＝200→1000 prefix・最初の比較 200 vs 400・seed spread ≤0.25・f64 primary＋f32 same-subset bound；距離は exact POT W₂）を **9 case すべて**について 1 run で実行し，case ごとの typed record（`cases/d2w_case_<F>_<S>.json`：W₂ manifest・stored evidence・decision・bank identity・入力束縛）と context record（`d2w_context_record.json`）を publication（内容検証付き）した後，**published record だけから context を復元**して live context と一致することを要求する（`G_records_restored`）。新しい bank・共分散・pseudo・較正・閾値・label は生成しない。入力は読取り専用。**標準 CPU runtime**（1 case の peak RSS ≈1.3 GB（sandbox 計測；K=50 の自己試験）；case は逐次；時間は実測で記録）。既定で各 run の `d2_bank_registry.json` と `cfg*_w2` だけを `STAGE_ROOT` に 1 回複製してから読む（複製前に disk 容量を検査；複製後も bytes／配列 SHA の再検証は同じ）。**3 family を 1 run** で処理する。

**attempt cell の規約（D-3c notebook v0.3 と同一；R-D3C1-A/launch-boundary・B/output-anchor）**：attempt cell は最初に **lock の出力先（anchor）** を検証し（symlink／非 directory／入力・source・stage root との交差／lock file の SHA 不一致なら何も書かずに停止），record はすべて anchor にだけ書く。設定・lock file・live source（実 HEAD・clean tree・inventory／script／pins／D-2 ledger／shared-null asset の file SHA・engine 版）の検査は 1 つの検査関数にまとめ，**staging の前**と**子 process 起動の直前**の 2 回呼ぶ。attempt ID を発行し，危険な処理の前に「未完了・非成功」の initial record を `d2w_final_record.json` に書く（前回の final record は `superseded_final_record_before_<attempt>.json` に退避し，今回の証拠として再利用しない）。`d2w_pass` は rc 0 ∧ `D2W_PASS` ∧ record の shape 正常 ∧ attempt／lock／source への束縛一致 ∧ `published_evidence` の全 file（9 case record＋context record）の SHA／bytes 一致 のときだけ True。科学的な結果（trigger／w2-unresolved）は PASS の条件ではない（技術 PASS と科学的 unresolved を区別する：R-D4DESIGN-B/E）。

**v0.2（監査 R-D4C0-A／B）**：(A) attempt cell の成功 record 検査を閉じる——schema 厳密一致・`stage=='complete'`・`failures==[]`・`selftest is False`・正式 profile・**REQUIRED の集合（13 名；信頼 source＝lock に束縛した script の `REQUIRED` 定数を AST で抽出し notebook 定数と一致を要求）と `required_inventory`・`gates` の key 集合・全値 `is True`・`required_all_true is True`**・**canonical 9 case（E2／E7／E8 × L1.00／L1.20／L1.50）の固定集合**（返却 cases から導かない；E1 等を認めない）・evidence file 集合はこの固定集合から導出・各 case record の内容（schema・key・family／size・formal・manifest／result SHA）と context record（context SHA・n_cases・case 要約）の run record との対応。科学的 w2-unresolved は技術 FAIL ではない（成功条件に含めない）。(B) preflight cell は **最初の mkdir／lock publication より前に出力先を検査**する（symlink・非 directory・source／入力／Phase C／stage／Drive root との交差・launcher 所有外の file を含む既存 directory を拒否；拒否先には何も書かない；安全な新規／空／launcher 所有の既存 directory のみ許容）；attempt cell も同じ規則（同じ関数）で anchor を再検査する。


In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the execution target (must contain d/d2w_cases.py and registered_assets/d2/d2_generation_ledger.json)>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
D2_RUN_ROOTS = {'E2': '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2',
                'E7': '/content/drive/MyDrive/MirrorTopology_D2/E7_20260923T125656Z/out/d2',
                'E8': '/content/drive/MyDrive/MirrorTopology_D2/E8_20260923T174910Z/out/d2'}   # the three accepted D-2 runs (out/d2 holding cfg<id>_w2 and d2_bank_registry.json)
STAGE_LOCAL = True                 # copy d2_bank_registry.json + the cfg*_w2 units of each run once to STAGE_ROOT before the intake (same verification; fewer Drive reads; disk capacity checked first)
STAGE_ROOT = '/content/stage'      # local staging directory (deleted and re-created by each attempt when STAGE_LOCAL)


In [ ]:
# --- 1. fresh scratch checkout; inventory + script + pins + D-2 ledger + shared-null asset bound; inputs exist and match the ledger; Drive mounted (read-only usage); the LOCK (published, hashed)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and set(D2_RUN_ROOTS)=={'E2','E7','E8'}
from google.colab import drive; drive.mount('/content/drive')
for f,p in D2_RUN_ROOTS.items(): assert os.path.isfile(f'{p}/d2_bank_registry.json'), p
SCRATCH='/content/d2w_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d2w_cases.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d2w_cases.py']
LEDGER=f'{PHASEB}/registered_assets/d2/d2_generation_ledger.json'; ASSET=f'{PHASEB}/registered_assets/b3_2_shared_null_asset.json'; PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(LEDGER)==inv['registered_assets_sha256']['registered_assets/d2/d2_generation_ledger.json'] and sha(ASSET)==inv['registered_assets_sha256']['registered_assets/b3_2_shared_null_asset.json'] and sha(PINS)==inv['d_sha256']['d/d3_pins.json']
ledger=json.load(open(LEDGER))
for f,p in D2_RUN_ROOTS.items(): assert p.rstrip('/')==ledger['families'][f]['run_root'].rstrip('/')+'/out/d2', (f, p, ledger['families'][f]['run_root'])
pins=json.load(open(PINS)); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1)); assert vm.total>10e9, 'at least a standard (>10 GB) runtime is required'
OUT='/content/d2w_cases'
# OUTPUT-ANCHOR (R-D4C0-B; the lines from here to the end of this cell are re-used verbatim by the contract test): the output directory is validated BEFORE the first mkdir / lock publication
_in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
OWNED=re.compile(r'(d2w_lock\.json|d2w_final_record\.json|superseded_final_record_before_[0-9TZ]+_[0-9a-f]{10}\.json|run_[0-9TZ]+_[0-9a-f]{10}|d2w_(stdout|stderr)_[0-9TZ]+_[0-9a-f]{10}\.txt)')
def _protected_roots(): return [MT, PHASEB, PHASEC, STAGE_ROOT, '/content/drive', *D2_RUN_ROOTS.values()]
def _safe_output_anchor(path, protected):
    # the ONE output policy (preflight and attempt): absolute path whose final component is not a symlink; disjoint from every protected root (source, Phase C, stage, Drive, inputs);
    # non-existent, or an existing real directory holding ONLY files owned by this launcher (lock, attempt records, run_* directories, stdout/stderr) and no symlinks. Nothing is written here.
    ap=os.path.abspath(path)
    if os.path.islink(ap) or os.path.realpath(ap)!=os.path.join(os.path.realpath(os.path.dirname(ap)), os.path.basename(ap)): raise RuntimeError(f'unsafe output: {path!r} is a symbolic link; nothing written')
    clash=[q for q in protected if _in(ap,q) or _in(q,ap)]
    if clash: raise RuntimeError(f'unsafe output: {path!r} overlaps a protected root {clash}; nothing written')
    if os.path.lexists(ap):
        if not os.path.isdir(ap): raise RuntimeError(f'unsafe output: {path!r} exists and is not a directory; nothing written')
        foreign=[e for e in sorted(os.listdir(ap)) if not OWNED.fullmatch(e) or os.path.islink(os.path.join(ap,e))]
        if foreign: raise RuntimeError(f'unsafe output: {path!r} holds entries not owned by this launcher {foreign[:5]}; nothing written')
    return ap
OUT=_safe_output_anchor(OUT, _protected_roots()); os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d2w_launcher_lock_v1', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d2w_cases.py'], ledger_file_sha256=sha(LEDGER), asset_file_sha256=sha(ASSET), pins_sha256=sha(PINS), engine=inv['engine_version'],
          d2_run_roots=dict(D2_RUN_ROOTS), stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/d2w_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print(lock)


In [ ]:
# --- 2. ONE ATTEMPT (D-3c v0.3 pattern; R-D3C1-A/launch-boundary + R-D3C1-B/output-anchor; R-D4DESIGN-G): (0) the SAFE anchored output is taken from the LOCK and validated BEFORE anything is written;
#        (1) attempt id + initial NON-success record at the anchor; (2) ONE verifier (configuration, lock file, LIVE source) runs BEFORE staging; (3) staging (disk check first; registry + cfg*_w2 only);
#        (4) the SAME verifier runs again IMMEDIATELY before subprocess.run (no child process on any mismatch); (5) record validation (trusted REQUIRED inventory, canonical cases) + binding + published-evidence content AND semantic check; final record at the anchor only
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
# 0. output anchor: from the in-memory lock (NOT the current OUT); re-validated with the SAME output policy as the preflight (_safe_output_anchor: no symlink, disjoint from the locked AND current
#    protected roots (source, Phase C, stage, Drive, inputs), existing real directory with launcher-owned entries only) and holding the published lock file; on refusal nothing is written anywhere
ANCHOR=lock['out']
try:
    if '_safe_output_anchor' not in globals(): raise RuntimeError('preflight cell not run')
    _safe_output_anchor(ANCHOR, [lock['mt'], lock['phaseb'], lock['phasec'], lock['stage_root'], '/content/drive', *lock['d2_run_roots'].values(), *_protected_roots()])
    if not os.path.isdir(ANCHOR) or LOCK_PATH!=f'{ANCHOR}/d2w_lock.json' or not os.path.isfile(LOCK_PATH): raise RuntimeError('lock path not inside the anchor')      # the lock file's CONTENT is verified by the verifier below (a FAIL record at this anchor)
except Exception as _anchor_error: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory ({_anchor_error}); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|D2W|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/d2w_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; d2w_pass=False; IN_D2=None; rc=None; v=None
REC=dict(schema='d2w_launcher_final_record_v2', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', d2w_pass=False, exit_code=None, D2W_PASS=None, launcher_fallback=False, gates_ok=None, bindings_ok=None, evidence_ok=None, failures=[], exception=None, run_dir=RUN, staged_inputs=None, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev      # a previous attempt's record is NEVER this attempt's evidence
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['d2w_pass']=bool(d2w_pass); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| d2w_pass', REC['d2w_pass'], '| failures', REC['failures'])
def _verify_locked_state(prefix):
    # the ONE verifier: current configuration == lock; lock file on disk == lock; LIVE source (actual HEAD, clean tree, file SHAs of inventory / script / pins / D-2 ledger / shared-null asset,
    # the inventory's script SHA, engine version, tree layout) == lock; input roots present. Called before staging and again immediately before the child launch.
    REC['stage']=f'{prefix}_config'; cfg=dict(d2_run_roots=dict(D2_RUN_ROOTS), stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/d2w_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d2w_cases.py'], ledger_file_sha256=sha(LEDGER), asset_file_sha256=sha(ASSET), pins_sha256=sha(PINS), engine=inv_now['engine_version'], checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], ledger_file_sha256=lock['ledger_file_sha256'], asset_file_sha256=lock['asset_file_sha256'], pins_sha256=lock['pins_sha256'], engine=lock['engine'])
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d2w_cases.py': raise RuntimeError(f'live source differs from the lock ({prefix}): {bad}')
    missing=[p for p in D2_RUN_ROOTS.values() if not os.path.isdir(p)]
    if missing: raise RuntimeError(f'input roots missing: {missing}')
try:
    # 2. configuration / lock file / live source BEFORE staging (editing cell 0 after the preflight, or skipping the preflight, is refused here)
    _verify_locked_state('precheck'); _publish(FIN, REC)
    # 3. optional local staging (registry + cfg*_w2 units only): disk capacity is checked BEFORE the copy; the script re-verifies every byte / array SHA of the staged copy
    REC['stage']='staging'; IN_D2=dict(D2_RUN_ROOTS)
    if STAGE_LOCAL:
        ST=STAGE_ROOT; guard=[q for q in [*D2_RUN_ROOTS.values(), ANCHOR, MT] if _in(ST,q) or _in(q,ST)]
        if guard: raise RuntimeError(f'STAGE_ROOT must be disjoint from the inputs, the output and the source: {guard}')
        units={f: sorted(u for u in os.listdir(p) if re.fullmatch(r'cfg\d+_w2', u) and os.path.isdir(os.path.join(p,u))) for f,p in D2_RUN_ROOTS.items()}
        need=sum(os.path.getsize(os.path.join(d,x)) for f,p in D2_RUN_ROOTS.items() for u in units[f] for d,_,fs in os.walk(os.path.join(p,u)) for x in fs)+sum(os.path.getsize(os.path.join(p,'d2_bank_registry.json')) for p in D2_RUN_ROOTS.values()); ts=time.time()
        shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST); free=shutil.disk_usage(ST).free; REC['staging']=dict(root=ST, units={f: len(u) for f,u in units.items()}, input_bytes=need, disk_free_bytes_before=free)
        if free < need*1.05+2e9: raise RuntimeError(f'insufficient local disk for staging: need {need/1e9:.1f} GB (+5% +2 GB margin), free {free/1e9:.1f} GB')
        IN_D2={f: f'{ST}/d2_{f}' for f in D2_RUN_ROOTS}
        for f,p in D2_RUN_ROOTS.items():
            os.makedirs(IN_D2[f]); shutil.copy2(os.path.join(p,'d2_bank_registry.json'), os.path.join(IN_D2[f],'d2_bank_registry.json'))
            for u in units[f]: shutil.copytree(os.path.join(p,u), os.path.join(IN_D2[f],u))
        REC['staging'].update(seconds=round(time.time()-ts,1), disk_free_bytes_after=shutil.disk_usage(ST).free); print('staged in', round(time.time()-ts), 's; disk free GB', round(REC['staging']['disk_free_bytes_after']/1e9,1))
    REC['staged_inputs']=dict(d2=IN_D2); _publish(FIN, REC)
    # 4. the SAME verifier IMMEDIATELY before the launch (state may have changed during the copy): any mismatch -> FAIL record, NO child process
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]+[x for f in sorted(IN_D2) for x in ('--d2-root', f'{f}={IN_D2[f]}')]
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/d2w_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/d2w_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    # 5. semantic validation of the record (R-D4C0-A): exact schema / complete stage / empty failures / formal profile / no self-test (shape -> launcher_fallback); the TRUSTED REQUIRED inventory
    #    (notebook constant == the REQUIRED tuple extracted by AST from the lock-bound script) with every gate `is True` and required_all_true (-> gates_ok); binding to THIS attempt / lock / source
    #    (-> bindings_ok); the CANONICAL nine cases (fixed constant, never derived from the record) with every published evidence file content-verified AND semantically bound (-> evidence_ok)
    REC['stage']='record'; vp=f'{RUN}/d2w_run_record.json'; shape=[]; gates_bad=[]; binding=[]; evidence=[]
    CANON=[f'{f}/{sz}' for f in ('E2','E7','E8') for sz in ('L1.00','L1.20','L1.50')]
    REQUIRED_EXPECTED=['G_pins_loaded','G_engine_inventory','G_script_sha','G_phaseC_members','G_env_lock','G_external_loader_sha','G_twelve_context','G_shared_null','G_d2_inputs_resolved','G_cases_assembled','G_context_built','G_records_restored','G_record_saved']
    import ast as _ast
    _req_src=[list(_ast.literal_eval(st.value)) for st in _ast.parse(open(SCRIPT).read()).body if isinstance(st,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='REQUIRED' for t in st.targets)]
    if _req_src!=[REQUIRED_EXPECTED]: gates_bad.append(f'REQUIRED inventory of the lock-bound script differs from the trusted constant: {_req_src}')
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'run record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('run record is not a dictionary'); v=None
    if v is not None:
        num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool)
        for k, ok in [('schema', v.get('schema')=='d2w_run_record_v1'), ('D2W_PASS', isinstance(v.get('D2W_PASS'), bool)), ('stage', v.get('stage')=='complete'), ('gates', isinstance(v.get('gates'), dict)), ('required_inventory', isinstance(v.get('required_inventory'), list)), ('required_all_true', isinstance(v.get('required_all_true'), bool)), ('failures', v.get('failures')==[]), ('seconds', num(v.get('seconds'))), ('stages_rss_mb', isinstance(v.get('stages_rss_mb'), dict)),
                      ('attempt', isinstance(v.get('attempt'), dict)), ('source', isinstance(v.get('source'), dict)), ('selftest', v.get('selftest') is False), ('profile', v.get('profile')=='production_official'), ('context', isinstance(v.get('context'), dict) and isinstance((v.get('context') or {}).get('cases'), dict)), ('published_evidence', isinstance(v.get('published_evidence'), dict))]:
            if not ok: shape.append(f'run record field invalid: {k}')
        g=v.get('gates') if isinstance(v.get('gates'), dict) else {}
        if sorted(g)!=sorted(REQUIRED_EXPECTED): gates_bad.append(f'gate set differs from the trusted REQUIRED inventory: {sorted(g)}')
        if v.get('required_inventory')!=REQUIRED_EXPECTED: gates_bad.append('required_inventory differs from the trusted REQUIRED inventory')
        if not all(g.get(k) is True for k in REQUIRED_EXPECTED): gates_bad.append(f'required gates not all True: {[k for k in REQUIRED_EXPECTED if g.get(k) is not True]}')
        if v.get('required_all_true') is not True: gates_bad.append('required_all_true is not True')
        a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}
        for k, ok in [('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine'])]:
            if not ok: binding.append(f'run record not bound to this attempt / lock: {k}')
        pe=v.get('published_evidence') if isinstance(v.get('published_evidence'), dict) else {}; ctx_=v.get('context') if isinstance(v.get('context'), dict) else {}; cases=ctx_.get('cases') if isinstance(ctx_.get('cases'), dict) else {}
        expected_files={f"d2w_case_{k.replace('/', '_')}.json" for k in CANON} | {'d2w_context_record.json'}
        if sorted(cases)!=sorted(CANON): evidence.append(f'case set differs from the canonical nine cases: {sorted(cases)}')
        if set(pe)!=expected_files: evidence.append(f'published evidence set {sorted(pe)} != canonical {sorted(expected_files)}')
        for fn in sorted(expected_files):
            e=pe.get(fn); fp=f'{RUN}/cases/{fn}' if fn.startswith('d2w_case_') else f'{RUN}/{fn}'
            if not (isinstance(e, dict) and os.path.isfile(fp) and not os.path.islink(fp) and sha(fp)==e.get('sha256') and os.path.getsize(fp)==e.get('bytes')): evidence.append(f'published evidence file not content-verified: {fn}'); continue
            try: doc=json.load(open(fp))
            except Exception as doc_error: evidence.append(f'published evidence file unreadable: {fn} {doc_error!r}'); continue
            if fn.startswith('d2w_case_'):
                k=[c for c in CANON if f"d2w_case_{c.replace('/', '_')}.json"==fn][0]; cs=cases.get(k) if isinstance(cases.get(k), dict) else {}; inp=doc.get('inputs') if isinstance(doc, dict) and isinstance(doc.get('inputs'), dict) else {}
                if not (isinstance(doc, dict) and doc.get('schema')=='d2w_case_record_v1' and doc.get('key')==k and (doc.get('family'), doc.get('size_id'))==tuple(k.split('/')) and inp.get('formal') is True and doc.get('manifest_sha256')==cs.get('manifest_sha256') and doc.get('result_sha256')==cs.get('result_sha256') and isinstance(doc.get('decision'), dict) and doc['decision'].get('trigger')==cs.get('trigger') and doc['decision'].get('validation_state')==cs.get('validation_state') and doc['decision'].get('B_final')==cs.get('B_final')): evidence.append(f'case record not bound to the run record summary: {fn}')
            else:
                cr=doc.get('case_records') if isinstance(doc, dict) and isinstance(doc.get('case_records'), dict) else {}
                if not (isinstance(doc, dict) and doc.get('schema')=='d2w_context_record_v1' and doc.get('context_sha256')==ctx_.get('context_sha256') and doc.get('asset_sha256')==ctx_.get('asset_sha256') and doc.get('n_cases')==9 and sorted(cr)==sorted(CANON) and all(isinstance(cr.get(k), dict) and cr[k].get('manifest_sha256')==(cases.get(k) or {}).get('manifest_sha256') and cr[k].get('result_sha256')==(cases.get(k) or {}).get('result_sha256') for k in CANON)): evidence.append('context record not bound to the run record summary')
        REC.update(D2W_PASS=v.get('D2W_PASS'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), stages_rss_mb=v.get('stages_rss_mb'), stages_peak_rss_mb=v.get('stages_peak_rss_mb'), seconds_script=v.get('seconds'), context_sha256=ctx_.get('context_sha256'), case_outcomes={k: dict(trigger=c.get('trigger'), validation_state=c.get('validation_state'), B_final=c.get('B_final')) for k,c in cases.items() if isinstance(c, dict)})
        print('rc', rc.returncode, '| stage', v.get('stage'), '| D2W_PASS', v.get('D2W_PASS'), '| failures', v.get('failures'), '| rss MB', v.get('stages_rss_mb'), '| peak rss MB', v.get('stages_peak_rss_mb'), '| seconds', v.get('seconds')); print('cases', REC['case_outcomes'])
    REC.update(launcher_fallback=bool(shape), gates_ok=(not gates_bad) if v is not None else None, bindings_ok=(not binding) if v is not None else None, evidence_ok=(not evidence) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    d2w_pass=bool(rc.returncode==0 and v is not None and v.get('D2W_PASS') is True and not shape and not gates_bad and not binding and not evidence)      # the scientific outcome (trigger / w2-unresolved) is NOT a condition
    _finish('complete', shape+gates_bad+binding+evidence)
except Exception as attempt_error:
    d2w_pass=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('d2w_pass', d2w_pass)


In [ ]:
# --- 3. zip the evidence (lock, attempt records (current + superseded), run record, the 9 case records + context record, stdout/stderr) for the audit and for registration
zp=f'/content/d2w_cases_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)